# Nghiên cứu & Phân tích Sàng lọc Bệnh Parkinson Qua Giọng Nói (Đồ án 2.1)
**Voice-Based Parkinson's Disease Screening Using Machine Learning**

Notebook này thực hiện:
1. Phân tích khám phá dữ liệu (EDA) trên tập dữ liệu UCI Parkinson's Voice Dataset.
2. Phân tích các đặc trưng giọng nói: Tần số cơ bản F0, Jitter (chu kỳ), Shimmer (biên độ), HNR (hài âm/nhiễu).
3. Huấn luyện và so sánh 4 mô hình Machine Learning: Logistic Regression, SVM, Random Forest, Gradient Boosting.
4. Đánh giá chi tiết: Accuracy, Precision, Recall, F1-score, ROC-AUC, Sensitivity, Specificity.
5. Minh họa trích xuất đặc trưng trực tiếp từ file âm thanh bằng module `audio_project.parkinson`.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Thêm đường dẫn project
sys.path.insert(0, '..')
from audio_project.parkinson import train_parkinson_model, extract_parkinson_features, UCI_FEATURES

print('Environment initialized successfully.')

## 1. Khám phá tập dữ liệu UCI Parkinson
Tập dữ liệu gồm 195 bản ghi âm từ 31 đối tượng (23 người bệnh Parkinson, 8 người khỏe mạnh).
Mỗi bản ghi chứa 22 thuộc tính âm học trích xuất từ âm nguyên âm kéo dài /a/.

In [ ]:
data_path = Path('../data/parkinsons.data')
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape}')
print(f'Healthy: {(df["status"] == 0).sum()} (status=0)')
print(f'Parkinson: {(df["status"] == 1).sum()} (status=1)')
df.head()

## 2. Huấn luyện và Đánh giá So sánh 4 Mô hình
So sánh Logistic Regression, SVM (RBF kernel), Random Forest và Gradient Boosting.
Sử dụng StratifiedKFold (k=5) và báo cáo đầy đủ Accuracy, Precision, Recall, F1, ROC-AUC, Sensitivity, Specificity.

In [ ]:
train_results = train_parkinson_model(data_path=str(data_path), verbose=True)
comp_df = pd.read_csv('../results/parkinson_model_comparison.csv')
comp_df

## 3. Trích xuất Đặc trưng từ File Âm thanh Trực tiếp (INMP441 / Microphone)
Module trích xuất đầy đủ 51 đặc trưng: F0 (mean, std, min, max, range), Jitter (local, rap, ppq5, ddp), Shimmer (local, db, apq3, apq5), HNR, NHR, MFCC 1-13 (mean & std), Spectral Centroid, Bandwidth, Rolloff, ZCR, RMS.

In [ ]:
# Demo trích xuất trên tín hiệu âm thanh mẫu (16 kHz, 3 giây)
sr = 16000
t = np.linspace(0, 3, 3 * sr)
test_signal = np.sin(2 * np.pi * 180 * t) + 0.1 * np.random.randn(len(t))
features = extract_parkinson_features(test_signal, sr=sr)
print(f'Extracted {len(features["feature_vector"])} features successfully.')
for k in ['f0_mean', 'f0_std', 'jitter_local', 'shimmer_local', 'hnr', 'nhr']:
    print(f'  {k}: {features[k]:.4f}')